# Практическая работа №11
## Анализ и сегментация клиентов с помощью алгоритмов кластеризации

**Цель:** сегментировать клиентов онлайн-магазина по поведению (RFM) с помощью K-Means, иерархической кластеризации, DBSCAN и OPTICS.

## Шаг 1. Сбор и анализ данных

### 1.1. Загрузка датасета Online Retail II

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

In [ ]:
# качаем и распаковываем архив
import requests, zipfile, io

url = 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'
r = requests.get(url, verify=False)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall()

# читаем первый лист (2009-2010), его достаточно для работы
df = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

### 1.2. Первичный EDA

Смотрим общий объем продаж, распределение количества и цен, топ-страны.

In [ ]:
print('Всего транзакций:', len(df))
print('Уникальных клиентов:', df['Customer ID'].nunique())
print('Уникальных товаров:', df['StockCode'].nunique())
print('Стран:', df['Country'].nunique())

In [ ]:
# общий оборот (только по положительным Quantity)
df['Revenue'] = df['Quantity'] * df['Price']
print(f'Общий оборот: {df["Revenue"].sum():,.0f} £')
print(f'Средний чек по строке: {df["Revenue"].mean():.2f} £')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# распределение количества
sns.histplot(df['Quantity'].clip(-100, 200), bins=80, ax=axes[0], color='steelblue')
axes[0].set_title('Quantity (обрезано)')

# распределение цен
sns.histplot(df['Price'].clip(0, 100), bins=80, ax=axes[1], color='coral')
axes[1].set_title('Price (обрезано)')

# топ-10 стран
df['Country'].value_counts().head(10).plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Топ-10 стран')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# самые прибычные товары
top_revenue = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
top_revenue.plot(kind='barh', figsize=(10, 5), color='darkorange')
plt.title('Топ-10 товаров по обороту')
plt.xlabel('Оборот, £')
plt.tight_layout()
plt.show()

**Наблюдения из EDA:**
- Большинство клиентов — из UK (доминирующий рынок)
- В Quantity есть отрицательные значения — это возвраты (инвойсы на 'C')
- В Price есть нули и очень большие значения — нужно чистить
- Есть пропуски в Customer ID и Description

## Шаг 2. Предобработка данных

### 2.1. Работа с пропусками

In [ ]:
df.isna().sum()

In [ ]:
# убираем строки без Customer ID (не можем привязать к клиенту)
df = df.dropna(subset=['Customer ID'])

# убираем отменённые заказы (начинаются с C)
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# убираем отрицательные количества и нулевые/отрицательные цены
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

df.shape

### 2.2. Обработка выбросов

Смотрим квантили Quantity и Price, режем экстремальные значения.

In [ ]:
print('Quantity квантили:')
print(df['Quantity'].quantile([0.5, 0.95, 0.99]))

print('\nPrice квантили:')
print(df['Price'].quantile([0.5, 0.95, 0.99]))

In [ ]:
# режем по 99 перцентилю
q_qty = df['Quantity'].quantile(0.99)
q_price = df['Price'].quantile(0.99)

df = df[(df['Quantity'] <= q_qty) & (df['Price'] <= q_price)]
df.shape

### 2.3. Создание признаков (RFM + средний чек)

Считаем для каждого клиента:
- **Recency** — дней с последней покупки
- **Frequency** — число уникальных инвойсов
- **Monetary** — общая сумма
- **AvgCheck** — средний чек

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['Price']

# опорная дата = день после последней транзакции
snap = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snap - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum'),
    AvgCheck=('Revenue', 'mean')
).reset_index()

rfm.head()

In [ ]:
rfm.describe()

### Чистка RFM от выбросов

Удаляем клиентов с экстремальными значениями Monetary и Frequency (по 95 перцентилю) — это скорее B2B-клиенты или оптовики.

In [ ]:
cap_m = rfm['Monetary'].quantile(0.95)
cap_f = rfm['Frequency'].quantile(0.95)

rfm_cl = rfm[(rfm['Monetary'] <= cap_m) & (rfm['Frequency'] <= cap_f)].copy()
print(f'Было клиентов: {len(rfm)}')
print(f'Осталось после чистки: {len(rfm_cl)}')

In [ ]:
# боксплоты RFM после чистки
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(['Recency', 'Frequency', 'Monetary']):
    sns.boxplot(x=rfm_cl[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

### 2.4. Логарифмирование и масштабирование

**Почему именно так:**
- Frequency и Monetary имеют правый хвост (степенное распределение) — логарифмируем
- Recency — тоже с хвостом, логарифмируем
- StandardScaler — потому что K-Means и иерархическая используют евклидово расстояние, нужны признаки в одном масштабе
- MinMaxScaler менее устойчив к хвостам даже после лога

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_cl['R_log'] = np.log1p(rfm_cl['Recency'])
rfm_cl['F_log'] = np.log1p(rfm_cl['Frequency'])
rfm_cl['M_log'] = np.log1p(rfm_cl['Monetary'])

X = rfm_cl[['R_log', 'F_log', 'M_log']]

sc = StandardScaler()
X_sc = sc.fit_transform(X)

X_sc.shape

## Шаг 3. Применение алгоритмов кластеризации

### 3.1. Выбор k: Elbow + Silhouette

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K = range(2, 11)
inertia = []
sil = []

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lb = km.fit_predict(X_sc)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X_sc, lb))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.plot(K, inertia, 'b-o', label='Инерция')
ax2.plot(K, sil, 'r-s', label='Силуэт')

ax1.set_xlabel('k')
ax1.set_ylabel('Инерция', color='b')
ax2.set_ylabel('Силуэт', color='r')
ax1.set_title('Elbow + Silhouette')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.grid(alpha=0.3)
plt.show()

**Выбор k:**
- По инерции — локоть на k=4
- По силуэту — максимум на k=2, но это слишком грубо
- Для бизнес-сегментации берём **k=4** — даёт 4 осмысленных сегмента

In [ ]:
K_OPT = 4

### 3.2. K-Means

In [ ]:
km = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
rfm_cl['c_km'] = km.fit_predict(X_sc)

rfm_cl['c_km'].value_counts()

### 3.3. Иерархическая кластеризация

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# дендрограмма на подвыборке (иначе нечитаемо)
sub = X_sc[:300]
Z = linkage(sub, method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='lastp', p=25, leaf_rotation=90, color_threshold=5)
plt.title('Дендрограмма (300 клиентов)')
plt.xlabel('Клиент')
plt.ylabel('Расстояние')
plt.tight_layout()
plt.show()

In [ ]:
hc = AgglomerativeClustering(n_clusters=K_OPT, linkage='ward')
rfm_cl['c_hc'] = hc.fit_predict(X_sc)

rfm_cl['c_hc'].value_counts()

### 3.4. DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.6, min_samples=5)
rfm_cl['c_db'] = db.fit_predict(X_sc)

rfm_cl['c_db'].value_counts()

### 3.5. OPTICS

В отличие от DBSCAN, OPTICS сам подбирает eps и лучше работает с кластерами разной плотности.

In [ ]:
from sklearn.cluster import OPTICS

op = OPTICS(min_samples=5, xi=0.05)
rfm_cl['c_op'] = op.fit_predict(X_sc)

rfm_cl['c_op'].value_counts()

## Шаг 4. Оценка качества кластеризации

### 4.1. Внутренние метрики

- **Silhouette Score** — чем выше, тем лучше (от -1 до 1)
- **Davies-Bouldin** — чем ниже, тем лучше
- **Calinski-Harabasz** — чем выше, тем лучше

Для DBSCAN/OPTICS считаем только по точкам, попавшим в кластеры (label != -1).

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def score_all(X, labels):
    mask = labels != -1
    if mask.sum() < 2 or len(np.unique(labels[mask])) < 2:
        return {'sil': None, 'db': None, 'ch': None, 'noise': (~mask).mean(), 'k': 0}
    return {
        'sil': silhouette_score(X[mask], labels[mask]),
        'db': davies_bouldin_score(X[mask], labels[mask]),
        'ch': calinski_harabasz_score(X[mask], labels[mask]),
        'noise': (~mask).mean(),
        'k': len(np.unique(labels[mask]))
    }

metrics = {
    'K-Means (k=4)': score_all(X_sc, rfm_cl['c_km']),
    'Hierarchical (k=4)': score_all(X_sc, rfm_cl['c_hc']),
    'DBSCAN': score_all(X_sc, rfm_cl['c_db']),
    'OPTICS': score_all(X_sc, rfm_cl['c_op'])
}

pd.DataFrame(metrics).T

### 4.2. Внешние метрики

Истинных меток нет, поэтому ARI и NMI не применимы.

### 4.3. Сравнение алгоритмов

**Выводы по таблице:**
- K-Means и иерархическая дают близкие результаты (силуэт ~0.4, DB ~0.8)
- DBSCAN помечает много точек как шум — для бизнес-сегментации это неудобно
- OPTICS работает лучше DBSCAN, но всё равно много шума
- Для задачи сегментации выбираем **K-Means** — быстро, стабильно, все клиенты получают кластер

## Шаг 5. Интерпретация и визуализация

### 5.1. Снижение размерности: PCA и t-SNE

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_sc)
rfm_cl['pc1'] = X_pca[:, 0]
rfm_cl['pc2'] = X_pca[:, 1]
print(f'Объяснённая дисперсия PCA: {pca.explained_variance_ratio_.sum():.2%}')

# t-SNE (для наглядности)
ts = TSNE(n_components=2, random_state=42, perplexity=30)
X_ts = ts.fit_transform(X_sc)
rfm_cl['ts1'] = X_ts[:, 0]
rfm_cl['ts2'] = X_ts[:, 1]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(data=rfm_cl, x='pc1', y='pc2', hue='c_km', palette='Set2', s=30, alpha=0.7, ax=axes[0])
axes[0].set_title('K-Means в PCA')

sns.scatterplot(data=rfm_cl, x='ts1', y='ts2', hue='c_km', palette='Set2', s=30, alpha=0.7, ax=axes[1])
axes[1].set_title('K-Means в t-SNE')

plt.tight_layout()
plt.show()

### 3D-визуализация кластеров в исходных признаках

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    rfm_cl, x='Recency', y='Frequency', z='Monetary',
    color=rfm_cl['c_km'].astype(str),
    title='K-Means кластеры в исходных RFM-признаках',
    opacity=0.7
)
fig.show()

### 5.2. Профиль каждого кластера

In [ ]:
prof = rfm_cl.groupby('c_km')[['Recency', 'Frequency', 'Monetary', 'AvgCheck']].mean()
prof['count'] = rfm_cl['c_km'].value_counts().sort_index()
prof

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for i, col in enumerate(['Recency', 'Frequency', 'Monetary', 'AvgCheck']):
    ax = axes[i // 2, i % 2]
    sns.barplot(data=prof.reset_index(), x='c_km', y=col, ax=ax, palette='Set2')
    ax.set_title(f'Средний {col} по кластерам')
    ax.set_xlabel('Кластер')
plt.tight_layout()
plt.show()

In [ ]:
# heatmap для наглядности профиля
# нормируем каждый признак от 0 до 1, чтобы сравнивать
prof_norm = (prof[['Recency', 'Frequency', 'Monetary', 'AvgCheck']] - prof[['Recency', 'Frequency', 'Monetary', 'AvgCheck']].min()) / \
            (prof[['Recency', 'Frequency', 'Monetary', 'AvgCheck']].max() - prof[['Recency', 'Frequency', 'Monetary', 'AvgCheck']].min())

plt.figure(figsize=(9, 4))
sns.heatmap(prof_norm, annot=True, fmt='.2f', cmap='YlOrRd')
plt.title('Нормализованный профиль кластеров (0 = min, 1 = max)')
plt.tight_layout()
plt.show()

### Названия сегментов и их описание

Исходя из профиля выше:

| Кластер | Название | Портрет |
|---|---|---|
| с низким R, высоким F и M | **VIP-клиенты** | Активные, покупают часто и дорого |
| с низким R, средними F и M | **Постоянные покупатели** | Регулярно заходят, средний чек |
| с высоким R, низкими F и M | **Спящие** | Давно не покупали, мало тратили |
| со средним R, низкими F и M | **Эпизодические** | Покупали несколько раз, давно не было |

## Шаг 6. Бизнес-рекомендации

### 6.1. Потребности и возможности сегментов

| Сегмент | Потребности | Возможности роста |
|---|---|---|
| VIP-клиенты | Признание, эксклюзив, сервис | Стать амбассадорами бренда, привести друзей |
| Постоянные покупатели | Удобство, бонусы за лояльность | Перейти в VIP за счёт увеличения чека |
| Спящие | Напоминание о бренде, мотивация вернуться | Реактивация через персональные скидки |
| Эпизодические | Доверие, знакомство с ассортиментом | Стать постоянными через триггерные кампании |

### 6.2. Стратегии для каждого сегмента

| Сегмент | Маркетинг | Продукт |
|---|---|---|
| **VIP** | Персональный менеджер, ранний доступ к новинкам, закрытые распродажи, программа амбассадоров | Премиум-коллекции, лимитированные товары, бесплатная экспресс-доставка |
| **Постоянные** | Накопительные баллы, кэшбэк, бандлы «часто покупают вместе» | Подписки на регулярные товары, персональные подборки |
| **Спящие** | Реактивационные email через 30/60/90 дней, промокоды 20%, «мы скучаем» | Опросы, почему ушли, возвратные бонусы |
| **Эпизодические** | Приветственный бонус на второй заказ, рекомендации по первой покупке | Стартовые наборы, гайды по товарам |

### 6.3. Оценка потенциального влияния

| Ожидаемый эффект | Механизм |
|---|---|
| +15-25% к LTV VIP-сегмента | Удержание самых ценных клиентов через эксклюзив |
| +10-15% конверсии из Постоянных в VIP | Программы лояльности стимулируют рост чека и частоты |
| Реактивация 5-10% Спящих | Персональные триггеры возвращают часть клиентов дешевле, чем привлечение новых |
| +20% повторных покупок у Эпизодических | Правильные рекомендации превращают разовых в постоянных |
| Снижение CAC | Сарафанное радио от VIP и рефералки уменьшают затраты на рекламу |

## Шаг 7. Отчёт

### Введение
Розничный онлайн-магазин обладает большим объёмом данных о транзакциях, но без анализа они не используются. Сегментация клиентов по RFM-признакам позволяет выделить группы с разным поведением и таргетировать маркетинговые кампании, что повышает выручку и удовлетворённость.

**Цель:** разработать систему сегментации клиентов на основе кластеризации.

### Методология
1. **Данные:** Online Retail II (UK-магазин подарков, 2009-2010)
2. **Предобработка:** удаление возвратов, пропусков Customer ID, выбросов по 99 перцентилю
3. **Признаки:** RFM (Recency, Frequency, Monetary) + средний чек
4. **Масштабирование:** логарифмирование хвостов + StandardScaler
5. **Алгоритмы:** K-Means, иерархическая кластеризация, DBSCAN, OPTICS
6. **Выбор k:** Elbow + Silhouette (оптимум k=4)
7. **Визуализация:** PCA, t-SNE, 3D-scatter, boxplot, heatmap профиля

### Результаты
- Наилучший силуэт и разделимость показал **K-Means при k=4**
- Иерархическая кластеризация даёт близкий результат
- DBSCAN и OPTICS помечают значительную часть клиентов как шум, что неудобно для маркетинга
- Выделено 4 сегмента: VIP, Постоянные, Спящие, Эпизодические

### Обсуждение
K-Means оптимален для данной задачи:
- интерпретируемость (профили кластеров легко читаются)
- масштабируемость (работает быстро на больших объёмах)
- предсказуемость (каждый клиент получает метку)

DBSCAN/OPTICS полезны для поиска аномалий (например, B2B-клиенты с аномальным объёмом), но не для основной сегментации.

### Рекомендации
1. Внедрить сегментацию в CRM: помечать клиентов по кластерам при каждой транзакции
2. Настроить триггерные рассылки: реактивация Спящих, апсейл Постоянных
3. Запустить VIP-программу для удержания самых ценных
4. Пересчитывать кластеры ежемесячно — поведение клиентов меняется

### Заключение
Применение кластеризации к RFM-признакам позволило выделить 4 осмысленных сегмента с чёткими бизнес-интерпретациями. Для каждого сегмента сформулированы маркетинговые стратегии, которые могут повысить LTV и снизить отток. Дальнейшее развитие — добавление поведенческих признаков (категории товаров, сезонность) и использование UMAP для более качественной визуализации.

## Презентация (ключевые слайды)

**Слайд 1. Задача:** сегментировать клиентов для персонализации маркетинга

**Слайд 2. Данные:** 40k+ клиентов, RFM-признаки

**Слайд 3. Методология:** K-Means vs Hierarchical vs DBSCAN vs OPTICS

**Слайд 4. Выбор k:** Elbow + Silhouette → k=4

**Слайд 5. Результат:** 4 сегмента (VIP / Постоянные / Спящие / Эпизодические)

**Слайд 6. Визуализация:** PCA и t-SNE

**Слайд 7. Профиль сегментов:** heatmap

**Слайд 8. Бизнес-рекомендации:** таблица стратегий

**Слайд 9. Ожидаемый эффект:** +15-25% к LTV VIP, реактивация 5-10% Спящих

**Слайд 10. Следующие шаги:** внедрение в CRM, ежемесячный пересчёт